# Phase 3 — Data Preparation

## 3.1 Define the Target Data Model & Cleaning Rules

We will establish:

1. Target transaction grain
2. Final column names
3. Final data types
4. Duplicate policy
5. Missing Customer ID policy
6. Missing Description policy
7. Country standardization policy
8. Transaction classification
9. Return/cancellation treatment
10. Non-merchandise treatment
11. Zero-price treatment
12. Negative-price treatment
13. Outlier treatment
14. Derived fields
15. Rules for the reporting dataset

### 3.1.1 Target Transaction Grain

The cleaned transaction dataset will retain the transaction-line grain of the
source data.

Each row represents one transaction line recorded in the original e-commerce
export.

An `Invoice` is therefore not treated as a unique row identifier because a
single invoice can contain multiple transaction lines.

This grain preserves the detail required for subsequent invoice-level,
product-level, customer-level, country-level, and monthly analysis.

### 3.1.2 Target Schema and Standardized Column Names

The cleaned transaction dataset will use a consistent lowercase `snake_case`
naming convention to improve readability and compatibility across Python,
SQL, Excel, and reporting workflows.

| Source Field  | Target Field   | Expected Type       | Description                               |
| ------------- | -------------- | ------------------- | ----------------------------------------- |
| `Invoice`     | `invoice`      | string              | Invoice/transaction identifier            |
| `StockCode`   | `stock_code`   | string              | Product or transaction code               |
| `Description` | `description`  | string              | Product or transaction description        |
| `Quantity`    | `quantity`     | integer             | Quantity recorded on the transaction line |
| `InvoiceDate` | `invoice_date` | datetime            | Date and time associated with the invoice |
| `Price`       | `unit_price`   | numeric             | Price recorded for the transaction line   |
| `Customer ID` | `customer_id`  | nullable identifier | Customer identifier where available       |
| `Country`     | `country`      | string              | Country associated with the transaction   |

`invoice` and `stock_code` will remain string-based identifiers because the
source data contains both numeric and alphanumeric values.

`customer_id` will be treated as an identifier rather than a numerical
measurement. Missing customer identifiers will be preserved until the
appropriate business rule is established.

Derived fields such as transaction classification, line value, reporting
value, and time-based reporting fields will be added later after their
business definitions have been established.


### 3.1.3 Duplicate-Record Policy

Duplicate handling will distinguish between repeated invoice identifiers and
actual duplicated transaction lines.

A repeated `invoice` value is not considered a duplicate because the source
data is recorded at transaction-line level and a single invoice may contain
multiple legitimate transaction lines.

Exact duplicate transaction rows are treated as potential duplicated records.
After the relevant fields have been standardized, identical transaction lines
will be evaluated using the duplicate rule defined for the cleaning pipeline.

For the cleaned analytical dataset, one instance of an exact duplicate
transaction line will be retained and additional identical instances will be
removed.

The pipeline will maintain an audit trail containing the raw row count,
duplicate rows identified, duplicate rows removed, and resulting cleaned row
count.

No duplicate handling is based solely on the `invoice` field.


### 3.1.4 Missing Customer ID Policy

`Customer ID` contains substantial missingness in the source data, with
20.54% of records missing the field in the 2009–2010 reporting period and
24.93% missing in the 2010–2011 reporting period.

Missing `customer_id` values will not be used as an automatic reason for
removing transaction records.

Transaction records with missing customer identifiers will be preserved in
the cleaned transaction dataset when they satisfy the other applicable data
quality and transaction-classification rules.

Missing customer identifiers will remain explicitly missing. Customer IDs
will not be invented, forward-filled, replaced with arbitrary placeholder
identifiers, or inferred from surrounding transactions.

Records with valid customer identifiers may be used for customer-level
analysis. Records without a valid customer identifier cannot be reliably
attributed to an individual customer and will therefore be excluded from
customer-specific attribution while remaining available for other applicable
transaction, product, time, or geographic analysis.

This policy preserves potentially valid business transactions while ensuring
that customer-level analysis does not imply information that is not supported
by the source data.


### 3.1.5 Missing Description Policy

The `Description` field contains a relatively small amount of missing data:
2,928 records (0.56%) in the 2009–2010 reporting period and 1,454 records
(0.27%) in the 2010–2011 reporting period.

Missing `description` values will not be used as an automatic reason for
removing transaction records.

The transaction will be preserved when it satisfies the other applicable data
quality and transaction-classification rules. The missing description will
remain explicitly missing rather than being replaced with an invented product
description.

Where appropriate, later stages may investigate whether a reliable
description can be associated with a transaction using other validated fields,
such as `stock_code` or an established transaction classification. Any such
standardization will be based on documented rules rather than assumptions.

The `description` field will therefore be treated as a descriptive attribute,
while `stock_code` remains the primary product/transaction code available for
product-level identification.
